# 03.04_Auco_rds_to_h5_R

Seurat RDS 转 H5 与元数据。

- 当前文件：`analysis/03_single_cell_processing/03.04_Auco_rds_to_h5_R.ipynb`
- 原始来源：`Codes/03.04_R_rds_to_h5.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`BiocManager`, `Matrix`, `Seurat`, `rhdf5`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。

**本文件说明：** 当前文件名含 Auco，正文仍包含 Auco/Trad 转换小节；按实际数据选择小节，未按新文件名删除代码。


r_scmap

# RDS to h5

In [ ]:
# if (!requireNamespace("BiocManager", quietly = TRUE))
#   install.packages("BiocManager")
# BiocManager::install("SeuratObject")
# BiocManager::install("Seurat")

library(Seurat)

# R code for converting RDS to h5
library(rhdf5)
library(Matrix)

# 保存为 h5 矩阵
save_h5mat = function(mat, fp_h5, feature_type, genome=""){
  # save sparse.mat ('dgCMatrix' format) into a h5 file
  # ======= Test code ======
  # tmp = Seurat::Read10X_h5(fp_h5)
  # all(tmp@x == mat@x)
  # all(tmp@i == mat@i)
  # all(tmp@p == mat@p)
  
  message(fp_h5)
  
  h5createFile(fp_h5)
  root = "matrix"
  h5createGroup(fp_h5, root)
  
  h5write(dim(mat), fp_h5, paste(root, "shape", sep='/'))
  h5write(mat@x, fp_h5, paste(root, "data", sep='/'))
  h5write(mat@i, fp_h5, paste(root, "indices", sep='/'))  # mat@i - 1 ?
  h5write(mat@p, fp_h5, paste(root, "indptr", sep='/'))
  h5write(colnames(mat), fp_h5, paste(root, "barcodes", sep='/'))
  
  
  feat_root = paste(root, "features", sep='/')
  h5createGroup(fp_h5, feat_root)
  
  h5write(rownames(mat), fp_h5, paste(feat_root, "id", sep='/'))
  h5write(rownames(mat), fp_h5, paste(feat_root, "name", sep='/'))
  
  h5write(rep(feature_type, dim(mat)[1]),
          fp_h5, paste(feat_root, "feature_type", sep='/'))
  
  h5write(rep("", dim(mat)[1]),
          fp_h5, paste(feat_root, "derivation", sep='/'))
  h5write(rep(genome, dim(mat)[1]),  # "mm10"
          fp_h5, paste(feat_root, "genome", sep='/'))
  h5write(c("genome", "derivation"),
          fp_h5, paste(feat_root, "_all_tag_keys", sep='/'))
  
  h5closeAll()
  message("Done!")
}

# save_h5mat_peak = function(mat, fp_h5, genome=""){
#   save_h5mat(mat, fp_h5, feature_type = "Peaks", genome = genome)
# }

save_h5mat_gex = function(mat, fp_h5, genome=""){
  save_h5mat(mat, fp_h5, feature_type = "Gene Expression", genome = genome)
}

## Auco

In [ ]:
## save the raw-counts in a Seurat-object "seurat_obj"
# 加载 RDS 数据
SCT_UMI_expression_matrix <- readRDS("/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix_CycloneSeq/analysis_results/Auco.seurat.rds")
seurat_object <- SCT_UMI_expression_matrix

In [ ]:
# 查看Seurat对象的基本信息
print(seurat_object)

In [ ]:
# 查看包含的 assays
Assays(seurat_object)

In [ ]:
# 查看元数据
head(seurat_object@meta.data)

In [ ]:
# ... [your existing code for save_h5mat_gex and loading seurat_object] ...

# --- Save Dimensional Reductions ---

# Check which reductions are available
print(Reductions(seurat_object)) # Should list 'pca', 'umap', 'tsne'

# Define base path for reductions
reduction_base_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix_CycloneSeq/sc_Auco.reduction."

# Save PCA coordinates
if ("pca" %in% Reductions(seurat_object)) {
  pca_coords <- Embeddings(seurat_object, reduction = "pca")
  write.csv(pca_coords, paste0(reduction_base_path, "pca.csv"), row.names = TRUE)
  message("PCA embeddings saved.")
}

# Save UMAP coordinates
if ("umap" %in% Reductions(seurat_object)) {
  umap_coords <- Embeddings(seurat_object, reduction = "umap")
  write.csv(umap_coords, paste0(reduction_base_path, "umap.csv"), row.names = TRUE)
  message("UMAP embeddings saved.")
}

# Save t-SNE coordinates
if ("tsne" %in% Reductions(seurat_object)) {
  tsne_coords <- Embeddings(seurat_object, reduction = "tsne")
  write.csv(tsne_coords, paste0(reduction_base_path, "tsne.csv"), row.names = TRUE)
  message("t-SNE embeddings saved.")
}

In [ ]:
# 定义输出文件路径模板
h5_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix_CycloneSeq/sc_Auco.matrix.raw.h5"
csv_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix_CycloneSeq/sc_Auco.metadata.csv"


# 提取计数矩阵, 对于 Assay5 类型，你需要使用 GetAssayData() 函数来提取原始计数矩阵：
mat <- GetAssayData(seurat_object, slot = "counts")
save_h5mat_gex(mat, h5_path, genome="")

# save the meta-data into a csv file:
meta_data = seurat_object@meta.data
write.csv(meta_data, csv_path)

## Trad

In [ ]:
## save the raw-counts in a Seurat-object "seurat_obj"
# 加载 RDS 数据
SCT_UMI_expression_matrix <- readRDS("/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/TR_scMatrix/analysis_results/Trad.seurat.rds")
seurat_object <- SCT_UMI_expression_matrix

In [ ]:
# 查看Seurat对象的基本信息
print(seurat_object)

In [ ]:
# 查看包含的 assays
Assays(seurat_object)

In [ ]:
# 查看元数据
head(seurat_object@meta.data)

In [ ]:
# ... [your existing code for save_h5mat_gex and loading seurat_object] ...

# --- Save Dimensional Reductions ---

# Check which reductions are available
print(Reductions(seurat_object)) # Should list 'pca', 'umap', 'tsne'

# Define base path for reductions
reduction_base_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/TR_scMatrix/sc_Trad.reduction."

# Save PCA coordinates
if ("pca" %in% Reductions(seurat_object)) {
  pca_coords <- Embeddings(seurat_object, reduction = "pca")
  write.csv(pca_coords, paste0(reduction_base_path, "pca.csv"), row.names = TRUE)
  message("PCA embeddings saved.")
}

# Save UMAP coordinates
if ("umap" %in% Reductions(seurat_object)) {
  umap_coords <- Embeddings(seurat_object, reduction = "umap")
  write.csv(umap_coords, paste0(reduction_base_path, "umap.csv"), row.names = TRUE)
  message("UMAP embeddings saved.")
}

# Save t-SNE coordinates
if ("tsne" %in% Reductions(seurat_object)) {
  tsne_coords <- Embeddings(seurat_object, reduction = "tsne")
  write.csv(tsne_coords, paste0(reduction_base_path, "tsne.csv"), row.names = TRUE)
  message("t-SNE embeddings saved.")
}

In [ ]:
# 定义输出文件路径模板
h5_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/TR_scMatrix/sc_Trad.matrix.raw.h5"
csv_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/TR_scMatrix/sc_Trad.metadata.csv"


# 提取计数矩阵, 对于 Assay5 类型，你需要使用 GetAssayData() 函数来提取原始计数矩阵：
mat <- GetAssayData(seurat_object, slot = "counts")
save_h5mat_gex(mat, h5_path, genome="")

# save the meta-data into a csv file:
meta_data = seurat_object@meta.data
write.csv(meta_data, csv_path)